### Purpose: Demonstrate Run Length Encoding (RLE) and how it's used by Parquet

In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import os

#### 1. RLE Implementation

In [2]:
def rle_encode(values):
    """Return [(value, run_length), ...]."""
    if not values:
        return []

    encoded = []
    current = values[0]
    count = 1

    for value in values[1:]:
        if value == current: # if next value is duplicate, increment count
            count += 1
        else:
            encoded.append((current, count)) # otherwise end the tuple
            current = value
            count = 1

    encoded.append((current, count))
    return encoded


def rle_decode(encoded):
    """Reconstruct the original sequence."""
    values = []

    for value, count in encoded:
        values.extend([value] * count)

    return values

#### 2. Create three different data patterns

In [3]:
repetitive = (
    ["US"] * 1000 +
    ["Spain"] * 1000 +
    ["France"] * 1000
)

moderate = (
    ["US", "US", "Spain", "Spain", "France", "France"] * 500
)

# Deterministic pattern without runs
no_runs_values = [
    ["US", "Canada", "UK", "France"][i % 4]
    for i in range(3000)
]

============================================================  

**Task 1: Test the `rle_encode()` function on each dataset**

============================================================  

#### 3. Look at the RLE representations

In [4]:
for name, values in [
    ("Highly repetitive", repetitive),
    ("Moderately repetitive", moderate),
    ("Random-looking", no_runs_values)
]:

    encoded = rle_encode(values)

    print("\n", name)
    print("-" * 50)
    print("Original values:", len(values))
    print("RLE runs:", len(encoded))
    print("Compression ratio:", len(values) / len(encoded))

    print("First 10 runs:")
    print(encoded[:10])


 Highly repetitive
--------------------------------------------------
Original values: 3000
RLE runs: 3
Compression ratio: 1000.0
First 10 runs:
[('US', 1000), ('Spain', 1000), ('France', 1000)]

 Moderately repetitive
--------------------------------------------------
Original values: 3000
RLE runs: 1500
Compression ratio: 2.0
First 10 runs:
[('US', 2), ('Spain', 2), ('France', 2), ('US', 2), ('Spain', 2), ('France', 2), ('US', 2), ('Spain', 2), ('France', 2), ('US', 2)]

 Random-looking
--------------------------------------------------
Original values: 3000
RLE runs: 3000
Compression ratio: 1.0
First 10 runs:
[('US', 1), ('Canada', 1), ('UK', 1), ('France', 1), ('US', 1), ('Canada', 1), ('UK', 1), ('France', 1), ('US', 1), ('Canada', 1)]


============================================================  

**Task 2: Notice the compression ratio from each dataset**

============================================================  

   - Notice details like compression ratio, row groups, and encodings


#### 4. Create Datasets and Write as Parquet

In [5]:
datasets = {
    "repetitive": repetitive,
    "moderate": moderate,
    "no_runs": no_runs_values
}

for name, values in datasets.items():

    df = pd.DataFrame({
        "status": values
    })

    # use pyarrow
    table = pa.Table.from_pandas(df)

    filename = f"{name}.parquet"

    # write as parquet file
    pq.write_table(
        table,
        filename,
        row_group_size=1000, # set row group size to partition into multiple groups
        use_dictionary=True
    )

    print(f"{filename}: {os.path.getsize(filename):,} bytes")

repetitive.parquet: 1,941 bytes
moderate.parquet: 2,046 bytes
no_runs.parquet: 2,049 bytes


#### 5. Inspect what Parquet Stored

In [6]:
for name in datasets:

    filename = f"{name}.parquet"

    pf = pq.ParquetFile(filename)

    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    print("Number of row groups:", pf.num_row_groups)

    for rg_number in range(pf.num_row_groups):

        rg = pf.metadata.row_group(rg_number)

        print("\nRow group:", rg_number)
        print("Rows:", rg.num_rows)

        for column_number in range(rg.num_columns):

            column = rg.column(column_number)

            print("Column:", column.path_in_schema)
            print("Encodings:", column.encodings)

            if column.statistics:
                print("Min:", column.statistics.min)
                print("Max:", column.statistics.max)
                print("Null count:", column.statistics.null_count)


REPETITIVE
Number of row groups: 3

Row group: 0
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: US
Max: US
Null count: 0

Row group: 1
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: Spain
Max: Spain
Null count: 0

Row group: 2
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: France
Max: France
Null count: 0

MODERATE
Number of row groups: 3

Row group: 0
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: France
Max: US
Null count: 0

Row group: 1
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: France
Max: US
Null count: 0

Row group: 2
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: France
Max: US
Null count: 0

NO_RUNS
Number of row groups: 3

Row group: 0
Rows: 1000
Column: status
Encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
Min: Canada
Max: US
Null count: 0

Row group: 1
Rows: 1000
Column: status
Encodings: ('PL

---

RLE_DICTIONARY creates a dictionary to map strings to IDs:

"United States" → 0  
"Spain"         → 1  
"France"        → 2

==============================================================================================  

**Task 3: Notice the row groups and encodings from each dataset. Does this info make sense?**

==============================================================================================  
